# Aula 1 · Notebook 04 — DuckDB: 3 modos de operação

Objetivo: entender que o **DuckDB não precisa de servidor nem de banco** — ele tem 3 formas de funcionar, e a mais usada é simplesmente **rodar SQL direto em arquivos**.

---

## O que tem aqui

| Modo | O que é | Quando usa |
|------|---------|-----------|
| 1 | Arquivo direto | Análise rápida (vamos usar muito) |
| 2 | Em memória | Tabelas temporárias durante a sessão |
| 3 | Arquivo persistente `.duckdb` | Como SQLite, guarda dados |

Tempo estimado: 15 minutos.

## Modo 1 — Direto em arquivo (90% dos casos)

Sem conexão, sem banco. Você dá o nome do arquivo no `FROM` da query.

In [ ]:
import duckdb

# Lendo direto do CSV — sem importar, sem criar tabela
resultado = duckdb.sql("""
    SELECT estado, COUNT(*) AS clientes, ROUND(SUM(valor_compra), 2) AS total
    FROM 'data/clientes_pequeno.csv'
    GROUP BY estado
    ORDER BY total DESC
""")
resultado.df()

Repare: **nenhum `connect()`**, **nenhum `INSERT`**, **nenhum `CREATE TABLE`**.

O arquivo *é* o banco. Você só pergunta.

## Modo 2 — Em memória

Quando você precisa criar tabelas temporárias durante a sessão Python (por exemplo, para JOINs intermediários).

In [ ]:
# Cria uma conexão na RAM
con = duckdb.connect()  # sem argumento = memória

# Cria uma tabela e popula
con.execute("""
    CREATE TABLE estados (sigla VARCHAR, regiao VARCHAR);
""")
con.execute("""
    INSERT INTO estados VALUES
        ('SP', 'Sudeste'),
        ('RJ', 'Sudeste'),
        ('PR', 'Sul'),
        ('RS', 'Sul'),
        ('BA', 'Nordeste');
""")

# Consulta a tabela em memória
con.sql("SELECT * FROM estados").df()

In [ ]:
# Você pode fazer JOIN entre tabela em memória e arquivo!
con.sql("""
    SELECT e.regiao, COUNT(*) AS clientes
    FROM 'data/clientes_pequeno.csv' c
    LEFT JOIN estados e ON c.estado = e.sigla
    GROUP BY e.regiao
    ORDER BY clientes DESC
""").df()

Quando o Python fechar, a tabela `estados` some. Vida útil = a sessão.

## Modo 3 — Arquivo persistente `.duckdb`

Quando você quer **guardar dados** entre execuções. Igual SQLite, mas otimizado para análise.

In [ ]:
import os

# Abrindo (ou criando) um banco persistente
con_p = duckdb.connect("data/meu_banco.duckdb")

# Cria tabela permanente
con_p.execute("""
    CREATE OR REPLACE TABLE clientes AS
    SELECT * FROM 'data/clientes_pequeno.csv'
""")

print("Tabela criada com",
      con_p.execute("SELECT COUNT(*) FROM clientes").fetchone()[0],
      "linhas")

con_p.close()

# Reabrindo depois — os dados continuam lá
con_p2 = duckdb.connect("data/meu_banco.duckdb")
total = con_p2.execute("SELECT COUNT(*) FROM clientes").fetchone()[0]
print(f"Banco persistido: {total} linhas ainda estão lá ✓")
con_p2.close()

# Limpando para o próximo notebook
os.remove("data/meu_banco.duckdb")

## Resumo

| Modo | Como conecta | Quando some |
|------|-------------|-------------|
| 1 — Arquivo direto | `duckdb.sql(...)` | Nada some |
| 2 — Em memória | `duckdb.connect()` | Quando o Python fecha |
| 3 — Persistente | `duckdb.connect("arquivo.duckdb")` | Nunca (vira arquivo) |

**No resto da aula vamos usar o Modo 1.**

## Mexa você mesmo

1. No Modo 1, escreva uma query que retorne **as 5 maiores compras de SP**.
2. No Modo 2, crie uma tabela `categorias_premium` com `('eletronicos'), ('moveis')` e faça um JOIN para contar quantos clientes compraram categorias premium.

In [ ]:
# Sua vez!


---

### Próximo notebook

[05 — DuckDB: SQL direto em arquivo](./05-duckdb-sql-em-arquivo.ipynb)

---
## Gabarito — Mexa você mesmo

In [ ]:
import duckdb

# 1. Modo 1 — as 5 maiores compras de SP (arquivo direto)
print("=== Top 5 maiores compras em SP ===")
duckdb.sql("""
    SELECT nome, cidade, valor_compra
    FROM 'data/clientes_pequeno.csv'
    WHERE estado = 'SP'
    ORDER BY valor_compra DESC
    LIMIT 5
""").df()

In [ ]:
# 2. Modo 2 — JOIN com tabela em memória
con = duckdb.connect()

# Cria a tabela de categorias premium
con.execute("""
    CREATE TABLE categorias_premium (categoria VARCHAR);
    INSERT INTO categorias_premium VALUES ('eletronicos'), ('moveis');
""")

# JOIN: clientes que compraram categorias premium
resultado = con.sql("""
    SELECT cp.categoria, COUNT(*) AS clientes
    FROM 'data/clientes_pequeno.csv' c
    INNER JOIN categorias_premium cp ON c.categoria = cp.categoria
    GROUP BY cp.categoria
    ORDER BY clientes DESC
""").df()

print("=== Clientes por categoria premium ===")
print(resultado)
con.close()